In [47]:
import pandas as pd 
import numpy as np 
import json 
import re 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
import configparser

In [48]:
config = configparser.ConfigParser()
config.read('../config.ini')

MAX_ITER = int(config['model']['max_iter'])
C = float(config['model']['C'])
MAX_FEATURES = int(config['model']['max_features'])
TEST_SIZE = float(config['model']['test_size'])
RANDOM_STATE = int(config['model']['random_state'])

print(f"max-iter = {MAX_ITER}, C = {C}, max-features = {MAX_FEATURES}, test-size = {TEST_SIZE}, random = {RANDOM_STATE}")


max-iter = 1000, C = 1.0, max-features = 10000, test-size = 0.2, random = 42


In [49]:
records = []
with open('../data/reviews_Digital_Music_5.json', 'r', encoding='utf=8') as f:
    for line in f:
        records.append(json.loads(line))
df = pd.DataFrame(records)

print(f"Колонки: {df.columns.tolist()}")
print(f"общее количество {len(df)}")




Колонки: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']
общее количество 64706


In [50]:

df = df.dropna(subset=['reviewText', 'overall'])

def get_sentiment(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else: 
        return 2

df['label'] = df['overall'].apply(get_sentiment)

def clean_text (text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df['clean_text'] = df['reviewText'].apply(clean_text)





In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
vectorizer =TfidfVectorizer(max_features=MAX_FEATURES)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
model = LogisticRegression(C=C, max_iter=MAX_ITER, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)


print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nDetailed report:")
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))


Accuracy: 0.8524

Detailed report:
              precision    recall  f1-score   support

    negative       0.76      0.47      0.58      1200
     neutral       0.53      0.21      0.30      1350
    positive       0.87      0.98      0.92     10392

    accuracy                           0.85     12942
   macro avg       0.72      0.55      0.60     12942
weighted avg       0.83      0.85      0.83     12942



In [ ]:
import os 

os.makedirs('../experiments', exist_ok=True)

joblib.dump(model, '../experiments/model.pkl')
joblib.dump(vectorizer, '../experiments/vectorizer.pkl')

loaded_model = joblib.load('../experiments/model.pkl')
loaded_vectorizer = joblib.load('../experiments/vectorizer.pkl')

test_reviews = [
    "This album is absolutely amazing, I love it!",
    "Terrible quality, waste of money.",
    "It's okay, nothing special."
]

for review in test_reviews:
    cleaned = clean_text(review)
    vec = loaded_vectorizer.transform([cleaned])
    pred = loaded_model.predict(vec)[0]
    labels = {0: 'negative', 1: 'neutral', 2: 'positive'}
    print(f"\n'{review}'\n→ {labels[pred]}")


TypeError: unhashable type: 'numpy.ndarray'